
# Likelihood Analysis — 4-Lineage Focus (No GLOBAL)

This notebook reads the CSV outputs under your `results/likelihood/tables/` folder and produces
**exhaustive** visual and tabular diagnostics with a strict focus on **four target lineages**.

**Key choices made here:**

- The **GLOBAL** pseudo-lineage is **excluded** from all visuals by default.
- You can **force exactly four lineages** (e.g., `B.1.1.7`, `B.1.351`, `P.1`, `B.1.617.2`).
  Anything else is optionally aggregated into an `"Other"` bucket (and the θ's are re-normalized).
- Adds deeper analyses: growth rates (logit slope), peak timing, cross-lineage correlations,
  leverage highlights, coverage-aware global mixtures, calibration curves, objective trace, etc.

Set the configuration in the next cell and run the notebook top-to-bottom.


In [ ]:

# === Configuration ===
import os

# Root where all likelihood outputs live
LIKELIHOOD_DIR = r"C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\likelihood"

# Subfolder where tables are stored (CSV + TEX files)
ALT_TABLES_DIR = os.path.join(LIKELIHOOD_DIR, "tables")

# Where this notebook will export figures and summary tables
OUTPUT_DIR = r"C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\likelihood_analysis_nb"

# Optional: regex to subset sites (e.g., r"^NYC|^SF"); set to None for all
SITE_FILTER_REGEX = None

# Plot and computation controls
TOP_LINEAGES = 8              # used only if EXACT_LINEAGES is None
MAX_SITES_PLOTS = 24          # limit per-site plots for large projects (in place of infinite)
DOWNSAMPLE_SCATTER = 200_000  # max hexbinned points in residual plots
SAVE_FIGS = False             # export PNGs + CSVs under OUTPUT_DIR
RANDOM_SEED = 12345           # reproducibility for downsampling

# ---- Strict 4-lineage focus ----
# If EXACT_LINEAGES is not None, we will keep only these names (case-sensitive matches against 'lineage' column),
# aggregate the rest into 'Other' (if AGGREGATE_OTHERS=True), drop 'GLOBAL', and re-normalize θ per sample.
# If any requested lineage is missing, we will auto-select the global Top-4 and print a warning.
EXACT_LINEAGES = ["B.1.1.7", "B.1.351", "P.1", "B.1.617.2"]
AGGREGATE_OTHERS = True
DROP_GLOBAL = True  # always recommended

# Optional pretty labels (only for plots); keys must match lineage names present in tables
RENAME_LINEAGES = {
    "B.1.1.7": "Alpha (B.1.1.7)",
    "B.1.351": "Beta (B.1.351)",
    "P.1": "Gamma (P.1)",
    "B.1.617.2": "Delta (B.1.617.2)",
    "Other": "Other (non-target)",
}


In [ ]:

import re
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

rng = np.random.default_rng(RANDOM_SEED)

def _ensure_dir(p: str):
    Path(p).mkdir(parents=True, exist_ok=True)

def _savefig(fig, name: str):
    if not SAVE_FIGS: 
        return
    _ensure_dir(OUTPUT_DIR)
    path = Path(OUTPUT_DIR) / f"{name}.png"
    fig.savefig(path, dpi=140, bbox_inches="tight")

def _savecsv(df: pd.DataFrame, name: str):
    if not SAVE_FIGS:
        return
    _ensure_dir(OUTPUT_DIR)
    path = Path(OUTPUT_DIR) / f"{name}.csv"
    df.to_csv(path, index=False)

def _maybe_pretty(name: str) -> str:
    return RENAME_LINEAGES.get(name, name)

def _subset_sites(df: pd.DataFrame, regex):
    if regex is None:
        return df.copy()
    m = df["site_id"].astype(str).str.contains(regex, case=False, regex=True)
    return df.loc[m].copy()

def _load_or_none(path):
    p = Path(path)
    return pd.read_csv(p) if p.exists() else None

# Robust table finder
TABLES = {
    "theta": "theta_estimates.csv",
    "theta_sd": "theta_uncertainty.csv",
    "residuals": "residuals.csv",
    "objective": "objective_trace.csv",
    "overlap": "overlap_matrix.csv",
    "signatures": "signatures_used.csv",
    "leverage": "mutation_leverage.csv",
    "zdiag": "zscore_diagnostics.csv",
    "simplex": "simplex_satisfied.csv",
}

def load_tables(root_dir: str) -> dict:
    tables = {}
    base = Path(root_dir)
    for k, fname in TABLES.items():
        p = base / fname
        tables[k] = _load_or_none(p)
    return tables

tables = load_tables(ALT_TABLES_DIR)
if tables["theta"] is None:
    raise FileNotFoundError(f"Missing required table: {ALT_TABLES_DIR}/theta_estimates.csv")
print("Loaded tables:")
for k,v in tables.items():
    print(f"  - {k:10s}: {'OK' if v is not None else 'missing'}")


In [ ]:

theta = tables["theta"].copy()
# Required columns sanity
need = {"site_id","date","sample_id","lineage","theta"} - set(theta.columns)
if need:
    raise ValueError(f"theta_estimates.csv missing columns: {need}")
theta["date"] = pd.to_datetime(theta["date"], errors="coerce")
theta = theta.dropna(subset=["date"])
theta["lineage"] = theta["lineage"].astype(str)
theta["site_id"] = theta["site_id"].astype(str)
theta["sample_id"] = theta["sample_id"].astype(str)

# Optional median_coverage may exist (added by the stage); if not, set to NaN
if "median_coverage" not in theta.columns:
    theta["median_coverage"] = np.nan

# Site subset (if requested)
theta = _subset_sites(theta, SITE_FILTER_REGEX)

# Drop GLOBAL if requested
if DROP_GLOBAL:
    theta = theta.loc[theta["lineage"].str.upper() != "GLOBAL"].copy()

# Detect available lineages
lineage_all = sorted(theta["lineage"].unique().tolist())

def pick_top4_by_global_share(th: pd.DataFrame) -> list:
    # Coverage-weighted mean per lineage across all sites/dates
    # If median_coverage is NaN, fall back to simple mean
    def _weighted_mean(g):
        w = g["median_coverage"].to_numpy(float)
        x = g["theta"].to_numpy(float)
        if np.all(~np.isfinite(w)) or np.nanmax(w) <= 0:
            return float(np.nanmean(x))
        w = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
        return float(np.sum(w * x) / max(np.sum(w), 1e-12))
    order = (th.groupby("lineage", as_index=False)
               .apply(_weighted_mean)
               .rename(columns={None:"wmean"})
               .sort_values("wmean", ascending=False))
    return order["lineage"].head(4).tolist()

selected = None
if EXACT_LINEAGES is not None:
    miss = [x for x in EXACT_LINEAGES if x not in lineage_all]
    if miss:
        print("WARNING: Some requested lineages not found:", miss)
        selected = pick_top4_by_global_share(theta)
        print("Auto-selected Top-4 lineages:", selected)
    else:
        selected = list(EXACT_LINEAGES)
else:
    selected = pick_top4_by_global_share(theta)
    print("No EXACT_LINEAGES specified; using Top-4:", selected)

# Build lineage_adj with 4 targets (+ optional Other)
if AGGREGATE_OTHERS:
    theta["lineage_adj"] = np.where(theta["lineage"].isin(selected), theta["lineage"], "Other")
else:
    theta["lineage_adj"] = theta["lineage"]
    # ensure only selected remain
    theta = theta.loc[theta["lineage_adj"].isin(selected)].copy()

# Re-normalize θ within each (site_id, date, sample_id)
grp_keys = ["site_id","date","sample_id"]
tot = (theta.groupby(grp_keys)["theta"].sum().rename("theta_sum").reset_index())
theta = theta.merge(tot, on=grp_keys, how="left")
theta["theta_norm"] = theta["theta"] / theta["theta_sum"].replace({0.0: np.nan})
theta["theta_norm"] = theta["theta_norm"].fillna(0.0).clip(0.0, 1.0)

# Collapse by lineage_adj after normalization
theta4 = (theta.groupby(grp_keys + ["lineage_adj"], as_index=False)["theta_norm"].sum()
               .rename(columns={"lineage_adj":"lineage", "theta_norm":"theta"}))

# Pretty labels for plotting (keep the raw 'lineage' for joins/exports)
theta4["lineage_label"] = theta4["lineage"].map(lambda x: RENAME_LINEAGES.get(x, x))

# Per-site, per-date average (if multiple samples per date)
site_date_mix = (theta4.groupby(["site_id","date","lineage"], as_index=False)["theta"].mean())

# Global (coverage-weighted) by date
# Join median_coverage per (site,date) if available in the original theta
med_cov = (theta.groupby(["site_id","date"], as_index=False)["median_coverage"].mean())
site_date_mix = site_date_mix.merge(med_cov, on=["site_id","date"], how="left")

def _global_weighted(df: pd.DataFrame) -> pd.DataFrame:
    out = []
    for (dt, lin), g in df.groupby(["date","lineage"], sort=False):
        w = g["median_coverage"].to_numpy(float)
        x = g["theta"].to_numpy(float)
        if np.all(~np.isfinite(w)) or np.nanmax(w) <= 0:
            val = float(np.nanmean(x))
        else:
            w = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
            val = float(np.sum(w * x) / max(np.sum(w), 1e-12))
        out.append({"date": pd.Timestamp(dt), "lineage": lin, "theta": val})
    return pd.DataFrame(out)

global_mix = _global_weighted(site_date_mix)
global_mix["lineage_label"] = global_mix["lineage"].map(lambda x: RENAME_LINEAGES.get(x, x))

# Export ready-to-plot tables
_savecsv(site_date_mix, "site_date_mixture_4lineages")
_savecsv(global_mix, "global_mixture_4lineages")

print("Prepared mixtures.")
print("Selected lineages:", selected, "(plus 'Other' aggregated)" if AGGREGATE_OTHERS else "")


In [ ]:

# --- Global stacked area ---
if global_mix.empty:
    print("No global mixture data to plot.")
else:
    dfp = global_mix.copy().sort_values("date")
    pivot = dfp.pivot(index="date", columns="lineage_label", values="theta").fillna(0.0)
    fig, ax = plt.subplots(figsize=(12, 6.5))
    ax.stackplot(pivot.index, pivot.T.values, labels=list(pivot.columns))
    ax.set_title("Global mixture (coverage-weighted) — 4 target lineages" + (" + Other" if "Other" in pivot.columns else ""))
    ax.set_ylabel("Share"); ax.set_xlabel("Date")
    ax.legend(loc="upper left", ncol=2, frameon=False)
    ax.set_ylim(0, 1.0)
    plt.tight_layout()
    _savefig(fig, "global_mixture_stack_4lineages")
    plt.show()

    # Last-date composition
    last_date = pivot.index.max()
    last = pivot.loc[last_date]
    last = last.sort_values(ascending=True)
    fig2, ax2 = plt.subplots(figsize=(7, 5))
    ax2.barh(last.index, last.values)
    ax2.set_title(f"Last-date composition — {last_date.date()}")
    ax2.set_xlabel("Share")
    plt.tight_layout()
    _savefig(fig2, "global_lastdate_composition_4lineages")
    plt.show()


In [ ]:

# --- Per-site stacked areas (first MAX_SITES_PLOTS sites) ---
sites = sorted(site_date_mix["site_id"].unique().tolist())
if not sites:
    print("No per-site mixture data available.")
else:
    shown = 0
    for site in sites:
        if shown >= MAX_SITES_PLOTS:
            break
        s = site_date_mix.loc[site_date_mix["site_id"] == site].copy()
        if s.empty: 
            continue
        pivot = (s.pivot_table(index="date", columns="lineage", values="theta", aggfunc="mean")
                   .sort_index().fillna(0.0))
        pivot = pivot.rename(columns=lambda x: RENAME_LINEAGES.get(x, x))
        fig, ax = plt.subplots(figsize=(12, 5.2))
        ax.stackplot(pivot.index, pivot.T.values, labels=list(pivot.columns))
        ax.set_title(f"{site} — mixture (4 target lineages" + (", + Other" if "Other" in pivot.columns else "") + ")")
        ax.set_ylabel("Share"); ax.set_xlabel("Date")
        ax.legend(loc="upper left", ncol=2, frameon=False)
        ax.set_ylim(0, 1.0)
        plt.tight_layout()
        _savefig(fig, f"site_{site}_mixture_stack")
        plt.show()
        shown += 1


In [ ]:

# --- Each lineage: cross-site lines (share vs date) ---
if site_date_mix.empty:
    print("No site-date mixture data.")
else:
    for lin, g in site_date_mix.groupby("lineage", sort=False):
        fig, ax = plt.subplots(figsize=(12, 5))
        for site, s in g.groupby("site_id", sort=False):
            s = s.sort_values("date")
            ax.plot(s["date"], s["theta"], alpha=0.7)
        ax.set_title(f"Cross-site trajectories: {_maybe_pretty(lin)}")
        ax.set_ylabel("Share"); ax.set_xlabel("Date")
        ax.set_ylim(0, 1.0)
        plt.tight_layout()
        _savefig(fig, f"cross_site_{lin}_trajectories")
        plt.show()


In [ ]:

res = tables["residuals"]
if res is None or res.empty:
    print("Residuals table not found — skipping residual diagnostics.")
else:
    df = res.copy()
    need = {"obs_af","pred_af","coverage"} - set(df.columns)
    if need:
        print("Residuals missing columns:", need)
    else:
        # Downsample for speed if needed
        if df.shape[0] > DOWNSAMPLE_SCATTER:
            df = df.loc[rng.choice(df.index.values, size=DOWNSAMPLE_SCATTER, replace=False)].copy()
        df = df.dropna(subset=["obs_af","pred_af"])

        # Pred vs Obs AF (hexbin)
        fig, ax = plt.subplots(figsize=(7.5, 6.2))
        hb = ax.hexbin(df["obs_af"], df["pred_af"], gridsize=55, mincnt=2)
        ax.plot([0,1], [0,1])
        ax.set_title("Predicted vs Observed AF")
        ax.set_xlabel("Observed AF"); ax.set_ylabel("Predicted AF")
        cb = fig.colorbar(hb, ax=ax); cb.set_label("count")
        plt.tight_layout(); _savefig(fig, "resid_pred_vs_obs_hex"); plt.show()

        # Residuals vs Coverage
        fig2, ax2 = plt.subplots(figsize=(7.8, 6.2))
        hb2 = ax2.hexbin(np.maximum(df["coverage"].to_numpy(float), 1.0),
                         (df["obs_af"] - df["pred_af"]).to_numpy(float),
                         gridsize=55)
        ax2.axhline(0.0)
        ax2.set_xscale("log")
        ax2.set_title("Residuals vs Coverage (log x)")
        ax2.set_xlabel("Coverage"); ax2.set_ylabel("obs - pred")
        fig2.colorbar(hb2, ax=ax2)
        plt.tight_layout(); _savefig(fig2, "resid_vs_coverage_hex"); plt.show()

        # Calibration curve (bin predicted AF)
        bins = np.linspace(0, 1, 26)
        df["pred_bin"] = pd.cut(df["pred_af"], bins, include_lowest=True)
        agg = df.groupby("pred_bin", as_index=False).agg(pred=("pred_af","mean"),
                                                         obs=("obs_af","mean"),
                                                         n=("obs_af","size"))
        fig3, ax3 = plt.subplots(figsize=(7.8, 6.0))
        ax3.plot([0,1], [0,1], "--")
        ax3.plot(agg["pred"], agg["obs"], marker="o", linestyle="-")
        ax3.set_title("Calibration (mean obs vs mean pred)")
        ax3.set_xlabel("Predicted AF"); ax3.set_ylabel("Observed AF")
        plt.tight_layout(); _savefig(fig3, "calibration_curve"); plt.show()

        _savecsv(agg, "calibration_bins")


In [ ]:

# --- Growth rates (approximate derivative of logit share) & peak timing ---
def _daily_grid(df: pd.DataFrame) -> pd.DataFrame:
    # ensure daily spacing for central differences
    all_dates = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    df = df.set_index("date").reindex(all_dates).interpolate().fillna(method="ffill").fillna(method="bfill")
    df = df.reset_index().rename(columns={"index":"date"})
    return df

def _logit(p, eps=1e-9):
    p = float(np.clip(p, eps, 1.0 - eps))
    return math.log(p/(1.0-p))

growth_rows = []
peak_rows = []
for (site, lin), g in site_date_mix.groupby(["site_id","lineage"], sort=False):
    s = g[["date","theta"]].sort_values("date").copy()
    if s.shape[0] < 3:
        continue
    s = _daily_grid(s)
    # central difference on logit(theta)
    vals = s["theta"].to_numpy(float)
    logits = np.array([_logit(x) for x in vals])
    dt = 1.0  # days
    dlogit = np.zeros_like(logits)
    dlogit[1:-1] = (logits[2:] - logits[:-2]) / (2*dt)
    dlogit[0] = logits[1] - logits[0]
    dlogit[-1] = logits[-1] - logits[-2]

    s["growth_logit_per_day"] = dlogit
    s["site_id"] = site; s["lineage"] = lin
    growth_rows.append(s[["site_id","date","lineage","theta","growth_logit_per_day"]])

    # peak timing
    idx = int(np.argmax(vals))
    peak_rows.append({"site_id": site, "lineage": lin, "peak_date": s["date"].iloc[idx], "peak_share": float(vals[idx])})

growth_df = pd.concat(growth_rows, ignore_index=True) if growth_rows else pd.DataFrame(columns=["site_id","date","lineage","theta","growth_logit_per_day"])
peaks_df  = pd.DataFrame(peak_rows) if peak_rows else pd.DataFrame(columns=["site_id","lineage","peak_date","peak_share"])

_savecsv(growth_df, "growth_rates_logit")
_savecsv(peaks_df, "lineage_peaks")

# Plot: global median growth per lineage
if not growth_df.empty:
    for lin, g in growth_df.groupby("lineage", sort=False):
        agg = g.groupby("date", as_index=False)["growth_logit_per_day"].median()
        fig, ax = plt.subplots(figsize=(10, 4.5))
        ax.plot(agg["date"], agg["growth_logit_per_day"])
        ax.axhline(0.0)
        ax.set_title(f"Median growth (logit/day): {_maybe_pretty(lin)}")
        ax.set_ylabel("d/dt logit(share)"); ax.set_xlabel("Date")
        plt.tight_layout(); _savefig(fig, f"growth_median_{lin}"); plt.show()

# Peak histograms
if not peaks_df.empty:
    for lin, g in peaks_df.groupby("lineage", sort=False):
        g = g.sort_values("peak_date")
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.hist(g["peak_date"].astype("int64")//10**9, bins=30)  # numeric bins
        ax.set_title(f"Peak date distribution: {_maybe_pretty(lin)}")
        ax.set_xlabel("Timestamp (s since epoch)"); ax.set_ylabel("count")
        plt.tight_layout(); _savefig(fig, f"peaks_hist_{lin}"); plt.show()


In [ ]:

# --- Cross-lineage correlations of share changes (within sites) ---
corr_rows = []
for site, g in site_date_mix.groupby("site_id", sort=False):
    pvt = g.pivot_table(index="date", columns="lineage", values="theta", aggfunc="mean").sort_index().fillna(method="ffill")
    if pvt.shape[0] < 3: 
        continue
    diffs = pvt.diff().dropna()
    C = diffs.corr()
    C["site_id"] = site
    corr_rows.append(C.reset_index().rename(columns={"index":"lineage"}))
corr_df = pd.concat(corr_rows, ignore_index=True) if corr_rows else pd.DataFrame()
_savecsv(corr_df, "cross_lineage_correlations_by_site")

if not corr_df.empty:
    # Average correlation matrix across sites
    cols = [c for c in corr_df.columns if c not in ("site_id","lineage")]
    avg = corr_df.groupby("lineage", as_index=False)[cols].mean().set_index("lineage")
    fig, ax = plt.subplots(figsize=(6, 5.2))
    im = ax.imshow(avg.values)
    ax.set_title("Average cross-lineage correlation (Δshare) across sites")
    ax.set_xticks(range(len(cols))); ax.set_xticklabels([_maybe_pretty(c) for c in cols], rotation=45, ha="right")
    ax.set_yticks(range(len(avg.index))); ax.set_yticklabels([_maybe_pretty(x) for x in avg.index])
    fig.colorbar(im, ax=ax)
    plt.tight_layout(); _savefig(fig, "avg_cross_lineage_corr"); plt.show()


In [ ]:

tsd = tables["theta_sd"]
if tsd is None or tsd.empty:
    print("theta_uncertainty table not found — skipping uncertainty bands.")
else:
    tsd = _subset_sites(tsd.copy(), SITE_FILTER_REGEX)
    tsd["date"] = pd.to_datetime(tsd["date"], errors="coerce")
    tsd = tsd.dropna(subset=["date"])
    tsd = tsd.loc[tsd["lineage"].isin(set(selected) | ({"Other"} if AGGREGATE_OTHERS else set()))].copy()

    # pick a representative site: most observations
    counts = tsd.groupby("site_id")["date"].nunique().sort_values(ascending=False)
    if counts.empty:
        print("No uncertainty data to plot.")
    else:
        site_rep = counts.index[0]
        print("Uncertainty bands for site:", site_rep)
        s_theta = site_date_mix.loc[site_date_mix["site_id"] == site_rep].copy()
        for lin in sorted(s_theta["lineage"].unique()):
            st = s_theta.loc[s_theta["lineage"] == lin].sort_values("date")
            sd = tsd[(tsd["site_id"] == site_rep) & (tsd["lineage"] == lin)].copy()
            if st.empty or sd.empty:
                continue
            # merge on nearest date (assume same dates ideally)
            m = pd.merge_asof(st.sort_values("date"), sd.sort_values("date"), on="date", direction="nearest")
            fig, ax = plt.subplots(figsize=(10, 4.5))
            ax.plot(m["date"], m["theta"], label=_maybe_pretty(lin))
            if "theta_sd" in m.columns:
                hi = (m["theta"] + 2*m["theta_sd"]).clip(0,1)
                lo = (m["theta"] - 2*m["theta_sd"]).clip(0,1)
                ax.fill_between(m["date"], lo, hi, alpha=0.25)
            ax.set_title(f"{site_rep} — {_maybe_pretty(lin)} (±2σ)")
            ax.set_ylabel("Share"); ax.set_xlabel("Date"); ax.set_ylim(0,1)
            ax.legend(frameon=False)
            plt.tight_layout(); _savefig(fig, f"uncertainty_{site_rep}_{lin}"); plt.show()


In [ ]:

lev = tables["leverage"]
if lev is None or lev.empty:
    print("Leverage table not found — skipping leverage view.")
else:
    # keep top rows by leverage
    lev2 = lev.copy()
    lev2 = _subset_sites(lev2, SITE_FILTER_REGEX)
    lev2["date"] = pd.to_datetime(lev2["date"], errors="coerce")
    lev2 = lev2.dropna(subset=["date"])
    top = lev2.sort_values("leverage", ascending=False).head(25)
    _savecsv(top, "top25_leverage_rows")
    print("Top-25 leverage rows:")
    display(top)


In [ ]:

O = tables["overlap"]
if O is None or O.empty:
    print("Overlap matrix not found — skipping.")
else:
    # Ensure we show only the selected lineages (+Other if present in columns)
    O = O.copy()
    if DROP_GLOBAL and "GLOBAL" in O.columns:
        O = O.drop(columns=[c for c in O.columns if c.upper() == "GLOBAL"], errors="ignore")
        O = O[O.index.str.upper() != "GLOBAL"]
    keep = set(selected) | ({"Other"} if AGGREGATE_OTHERS else set())
    cols = [c for c in O.columns if c in keep]
    idxs = [i for i in O.index if i in keep]
    if cols and idxs:
        O2 = O.loc[idxs, cols]
        fig, ax = plt.subplots(figsize=(6.5, 5.8))
        im = ax.imshow(np.clip(O2.values, 0.0, 1.0))
        ax.set_title("Signature column overlap")
        ax.set_xticks(range(len(cols))); ax.set_xticklabels([_maybe_pretty(c) for c in cols], rotation=45, ha="right")
        ax.set_yticks(range(len(idxs))); ax.set_yticklabels([_maybe_pretty(i) for i in idxs])
        fig.colorbar(im, ax=ax)
        plt.tight_layout(); _savefig(fig, "overlap_selected"); plt.show()
    else:
        print("No overlap columns match selected lineages; skipping.")


In [ ]:

obj = tables["objective"]
if obj is None or obj.empty:
    print("Objective trace not found — skipping.")
else:
    obj = _subset_sites(obj.copy(), SITE_FILTER_REGEX)
    if {"site_id","iter","objective"} <= set(obj.columns):
        # Per-site
        for site, g in obj.groupby("site_id", sort=False):
            fig, ax = plt.subplots(figsize=(7.5, 4.5))
            ax.plot(g["iter"], g["objective"])
            ax.set_title(f"Objective trace — {site}")
            ax.set_xlabel("iteration"); ax.set_ylabel("objective")
            plt.tight_layout(); _savefig(fig, f"objective_{site}"); plt.show()
        # Aggregate (median over sites per iter)
        agg = obj.groupby("iter", as_index=False)["objective"].median()
        fig2, ax2 = plt.subplots(figsize=(7.5, 4.5))
        ax2.plot(agg["iter"], agg["objective"])
        ax2.set_title("Objective (median across sites)")
        ax2.set_xlabel("iteration"); ax2.set_ylabel("objective")
        plt.tight_layout(); _savefig(fig2, "objective_median"); plt.show()
    else:
        print("Objective table lacks required columns; skipping.")


In [ ]:

print("=== Summary ===")
print(f"Tables root: {ALT_TABLES_DIR}")
print(f"Selected 4 lineages: {selected}")
if AGGREGATE_OTHERS:
    print("Other lineages aggregated into 'Other' (θ re-normalized per sample).")
print(f"SITE_FILTER_REGEX: {SITE_FILTER_REGEX}")
print(f"Exports written under: {OUTPUT_DIR if SAVE_FIGS else '(SAVE_FIGS=False; no files written)'}")
